# **Maestría en Ingteligencia Artificial Aplicada**
## **Procesamiento de Lenguaje Natural PLN-NLP**

## Ejercicio para el ajuste fino (fine-tuning) de un modelo LLM con QLoRA

#### Prof Luis Eduardo Falcón Morales

#### **Tecnológico de Monterrey**

**Usaremos la base de datos de HuggingFace llamada *cnn_dailynews*, en la cual una columna contiene la noticia escrita por un periodista de CNN y la otra columna es un resumen de dicha noticia hecha por el mismo periodista.**

  * **El objetivo de este ejercicio es entrenar con ajsute fino (fine-tuning) a un modelo LLM para que sepa generar resúmenes  de noticias como los periodistas de la cadena de noticias CNN.**

  * **Observa que en este caso no necesitaremos generar una base de datos vectorial, ya que no requeriremos recuperar información externa al momento de generar un resumen con el modelo final. Es decir, este no es un sistema RAG.**

  * **Recuerda usar GPU. En general podemos usar los recursos sin costo de Colab con la GPU-T4 que nos proporciona. Con mejores GPU en el caso de paga se pueden tener menores tiempos de entrenamiento.**

* **El pipeline de este ejercicio quedaría como sigue:**

**datos** $⇒$ **LLM cuantizado_Q_4-bit** $⇒$ **LoRA (fine-tuning)** $⇒$ **prompt + target** $⇒$ **Entrenamiento trainer()** $⇒$ **Evaluación: ROUGE / BERT_score**

https://huggingface.co/datasets/abisee/cnn_dailymail

In [ ]:
# Si deseas fijar versiones:
#!pip install transformers==4.40.2 accelerate==0.29.3 peft==0.10.0 bitsandbytes==0.43.1

In [ ]:
# O bien, instalar con las actualizadas: ... debes reiniciar después de instalarlas:

!pip install -U transformers accelerate datasets peft bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 115.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.6 MB/s eta 0:00:00


In [ ]:
!pip install -q evaluate rouge_score bert_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.0 MB/s eta 0:00:00


In [ ]:
# Para cargar los datos y modelos de HF:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# Para la cuantización-Q:
import torch
from transformers import BitsAndBytesConfig

# Para LoRA:
from peft import LoraConfig, get_peft_model

# Para configuración de los parámetros a ajustar
# y entrenar del LLM:
from transformers import TrainingArguments, Trainer

# Para la evaluación con ROUGE, BERT-score:
import evaluate
from bert_score import score as bertscore

### **Preparación de Datos**

In [ ]:
# Cargaremos el dataset CNN/DailyMail desde HuggingFace.
# Observa que no se está descargando la base de datos.

# Deberás dar acceso a tu HF_TOKEN para usar la base de datos de HuggingFace:

# Para no perder los datos con texto originales después de generar los embebidos definimos estos raw:
raw_dataset = load_dataset("abisee/cnn_dailymail", "3.0.0").shuffle(seed=17)  # Para evitar cualquier patrón en el cual están guardados.
raw_dataset.shape


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

{'train': (287113, 3), 'validation': (13368, 3), 'test': (11490, 3)}

In [ ]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [ ]:
# Desplegamos el primer dato:
raw_dataset['train'][0]

{'article': 'By . Ben Ellery . PUBLISHED: . 17:04 EST, 23 February 2013 . | . UPDATED: . 17:09 EST, 23 February 2013 . Too posh: Listeners have criticised Archers actress Heather Bell . The editor of The Archers has sprung to the defence of actress Heather Bell, who has returned to the role of Clarrie Grundy in the Radio 4 show, amid claims that she is ‘too posh’. Other listeners posting comments online complain that Ms Bell, 68, sounds like ‘an over-excited Pam Ayres’, while some allege that her accent is ‘wobbly’. Ms Bell played the character 25 years ago before being replaced by Rosalind Adams. However, she returned to Ambridge earlier this month. One Twitter user wrote: ‘They seem to have replaced Clarrie with an over-excited Pam Ayres. I hope she remembers to breathe.’ Such is the criticism aimed at Ms Bell that The Archers’ editor, Vanessa Whitburn, has now released a statement backing the actress. She said: ‘Many listeners are delighted to hear the original Clarrie again. Heathe

In [ ]:
# Tomemos solo una muestra aleatoria para los fines de esta actividad.

train_raw = raw_dataset["train"].select(range(50))    # 20_000 ... aún con GPU, puede tardar un par de horas con 20 mil.
val_raw = raw_dataset["validation"].select(range(10))  # 4_000

print(train_raw.shape)
print(val_raw.shape)

(50, 3)
(10, 3)


In [38]:
train_raw

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 50
})

In [39]:
print(train_raw[0]["article"])
print(" ")
print(train_raw[0]["highlights"])

By . Ben Ellery . PUBLISHED: . 17:04 EST, 23 February 2013 . | . UPDATED: . 17:09 EST, 23 February 2013 . Too posh: Listeners have criticised Archers actress Heather Bell . The editor of The Archers has sprung to the defence of actress Heather Bell, who has returned to the role of Clarrie Grundy in the Radio 4 show, amid claims that she is ‘too posh’. Other listeners posting comments online complain that Ms Bell, 68, sounds like ‘an over-excited Pam Ayres’, while some allege that her accent is ‘wobbly’. Ms Bell played the character 25 years ago before being replaced by Rosalind Adams. However, she returned to Ambridge earlier this month. One Twitter user wrote: ‘They seem to have replaced Clarrie with an over-excited Pam Ayres. I hope she remembers to breathe.’ Such is the criticism aimed at Ms Bell that The Archers’ editor, Vanessa Whitburn, has now released a statement backing the actress. She said: ‘Many listeners are delighted to hear the original Clarrie again. Heather is doing a 

### **LLM - cuantización 4-bit Q**

In [40]:
# Usaremos uno de los modelos Mistral de 7B para inglés bastante aceptable
# y de los que sabe seguir instrucciones, que para nuestro problema, promete
# ser de mayor ayuda que solo el modelo base.
# En ocasiones este modelo base suele tomarse como el modelo inicial, base
# o ingenuo precisamente, que trataremos de superar con modelos más grandes:

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Recordemos que Mistral no tiene [PAD] token.



In [41]:
# Cargamos el modelo cuantizado con 4-bit:

# Configuramos los argumentos a formato 4-bit:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Cargamos el modelo cuantizado.
# Aún con cuantización puede subir mucho la RAM
# dependiendo del LLM utilizado.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"   # detecta si hay GPU para utilizarlo
)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


### **Configuración de LoRA para el ajuste fino del LLM**

In [42]:
# Configuremos LoRA con ajuste en este caso
# solo de las matrices Q y V:

lora_config = LoraConfig(
    r=8,   # RAM
    lora_alpha=16,   # RAM
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


# Lo aplicamos a nuestro LLM:
model = get_peft_model(model, lora_config)

# Para cuidar que no nos truene la memoria activamos
# este checkpoint del gradiente, lo cual recalcula activaciones.
# Sin el checkpoint se guardan todas las activaciones en cache de memoria
# y eso lo hace más rápido, pero truena si no tienes muchos recursos.
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False

# Mostramos la cantidad de pesos o parámetros que se van a ajustar.
# Recuerda que en general se espera que el ajuste
# sea abajo del 1%, de lo contrario empezarás a perder poder del LLM:
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470


* **Recomendaciones para [r, alpha] de LoRA:**

  * alpha ≈ 2 × r
  * Con modelos pequeños <3B : [8, 16]
  * Con modelos medianos 7B aprox: [8-16, 16-32]
  * Con modelos grandes 12+: [16-32, 32-64]

* **r más grandes requieren más RAM.**
* **alpha más grande realiza ajsutas más fuertes, pero podría generar sobre-ajuste.**

* **En el ajuste fino de los pesos de matrices de Attention[Q,K,V] con LoRA:**

  * **Q(query): ¿Qué buscas?** Controla la palabra/token de referencia para buscar cómo se relaciona con las demás y con cuáles debe prestar más atención.
  * **K(key): ¿Qué características tengo?** Evalúa la información que tiene cada token mediante la similaridad con todas las demás.
  * **V(value): ¿Qué información tengo?** Nos da el contenido real de cada token.

    * **["q_proj", "v_proj"]** $⇒$ Por experiencia son los que generan mayores cambios.
    * **["q_proj", "k_proj", "v_proj", "o_proj"]** $⇒$ Si requieres mayor calidad, pero también mayores recursos.

* **bias="none":** En general no ajustarlos. Salvo ajustes muy fuertes o con datasets pequeños, usar "lora_only".

* **En el caso de "task_type" se indica el tipo de modelo que se va a ajustar (fine-tuning):**
  * **CAUSAL_LM:** Para modelos unidireccionales generativos, tipo GPT, indicando que se generan tokens uno a uno.
  * **SEQ_2_SEQ_LM:** Para modelos bidireccionales tipo BERT, T5.
  * **TOKEN_CLASSIFICATION:** Para NER.
  * **SEQ_CLS:** Para clasificación.


In [43]:
# Definamos la función que dará forma al prompt con la información
# de las dos primeras columnas de nuestros datos:

def format_example(example):
    prompt = f"""Summarize the following news article into bullet-point highlights:

Article:
{example['article']}

Highlights:
"""

    target = example["highlights"]

    full_text = prompt + target

    tokenized = tokenizer(
        full_text,
        truncation=True,
        max_length=512,  # RAM
        padding="max_length"
    )

    input_ids = tokenized["input_ids"]

    # máscara para NO entrenar/ajustar el prompt:
    prompt_ids = tokenizer(
        prompt,
        truncation=True,
        max_length=512   # RAM
    )["input_ids"]

    labels = input_ids.copy()
    labels[:len(prompt_ids)] = [-100] * len(prompt_ids)  # aquí estamos enmascarando el prompt
                                                         # para no aprenderlo/ajustarlo durante el entenamiento.
    tokenized["labels"] = labels

    return tokenized

In [44]:
# Y ahora sí, definimos los que se van a transformar a embbedings.
# Como vamos a quitar los nombres de las columnas para el proceso
# de entrenamiento, aún tendremos los datos "raw" para más adelante
# recuperar los nombres de las columnas al evaluar el modelo.

train_data = train_raw.map(format_example, remove_columns=train_raw.column_names)
val_data = val_raw.map(format_example, remove_columns=val_raw.column_names)

In [ ]:
# Desplegamos algunas de los primeros datos:

train_data[:1]

{'input_ids': [[1,
   6927,
   3479,
   653,
   272,
   2296,
   4231,
   5447,
   778,
   17144,
   28733,
   2275,
   23089,
   28747,
   13,
   13,
   10363,
   2660,
   28747,
   13,
   1930,
   842,
   4121,
   7205,
   1193,
   842,
   367,
   6870,
   28758,
   1851,
   22035,
   28747,
   842,
   28705,
   28740,
   28787,
   28747,
   28734,
   28781,
   413,
   920,
   28725,
   28705,
   28750,
   28770,
   5353,
   28705,
   28750,
   28734,
   28740,
   28770,
   842,
   342,
   842,
   500,
   8268,
   14371,
   28747,
   842,
   28705,
   28740,
   28787,
   28747,
   28734,
   28774,
   413,
   920,
   28725,
   28705,
   28750,
   28770,
   5353,
   28705,
   28750,
   28734,
   28740,
   28770,
   842,
   16601,
   977,
   28716,
   28747,
   26756,
   404,
   506,
   8394,
   2458,
   6573,
   404,
   18334,
   650,
   1223,
   11395,
   842,
   415,
   7546,
   302,
   415,
   6573,
   404,
   659,
   7378,
   969,
   298,
   272,
   22763,
   302,
   18334,
   650,

In [45]:
# Tokenizamos los datos con el formato prompt+target:

train_data = train_raw.map(format_example, remove_columns=train_raw.column_names)
val_data = val_raw.map(format_example, remove_columns=val_raw.column_names)

In [ ]:
train_data

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 50
})

In [ ]:
train_data[0]

{'input_ids': [1,
  6927,
  3479,
  653,
  272,
  2296,
  4231,
  5447,
  778,
  17144,
  28733,
  2275,
  23089,
  28747,
  13,
  13,
  10363,
  2660,
  28747,
  13,
  1930,
  842,
  4121,
  7205,
  1193,
  842,
  367,
  6870,
  28758,
  1851,
  22035,
  28747,
  842,
  28705,
  28740,
  28787,
  28747,
  28734,
  28781,
  413,
  920,
  28725,
  28705,
  28750,
  28770,
  5353,
  28705,
  28750,
  28734,
  28740,
  28770,
  842,
  342,
  842,
  500,
  8268,
  14371,
  28747,
  842,
  28705,
  28740,
  28787,
  28747,
  28734,
  28774,
  413,
  920,
  28725,
  28705,
  28750,
  28770,
  5353,
  28705,
  28750,
  28734,
  28740,
  28770,
  842,
  16601,
  977,
  28716,
  28747,
  26756,
  404,
  506,
  8394,
  2458,
  6573,
  404,
  18334,
  650,
  1223,
  11395,
  842,
  415,
  7546,
  302,
  415,
  6573,
  404,
  659,
  7378,
  969,
  298,
  272,
  22763,
  302,
  18334,
  650,
  1223,
  11395,
  28725,
  693,
  659,
  4253,
  298,
  272,
  3905,
  302,
  20225,
  3552,
  17583,
  287

## **Si al llevar a cabo el entrenamiento o en algún momento te truena la memoria, puedes probar las siguientes alternativas:**

1. Bajar el batch real (principal causa):
    * per_device_train_batch_size=1   # simula 1x8 registros de feedforward/backpropagation
    * gradient_accumulation_steps=8   # Con 4x4=16 requieres más memoria

2. Reducir la longitud de "article + prompt + highlights". Este puede subir mucho la RAM porque es O(n^2) en LLMs:
    * max_lengt=512   # grande 1024, tal vez 768, o más pequeño 256

3. Activar el gradient checkpoint (clave con LLMs):
    * model.gradient_checkpointing_enable()
    * model.config.use_cache = False

4. Ajustar la configuración de QLoRA:
    * r=8   #  con mayores recursos 16, o bien 4
    * lora_alpha=16   #  32, o bien 8

5. Cargar un LLM más pequeño.

6. Cargar conjuntos Train y Val más pequeños.

7. Si ya te tronó la memoria, y no deseas empezar de cero, libera la memoria con la siguiente instrucción y luego reinicia la sesión (runtime) ... aunque no siempre puede funcionar :S
    * import torch
    * torch.cuda.empty_cache()

8. La siguiente fórmula nos ayuda a simular un batch grande con RAM pequeña, porque recuerda que los LLM requieren "batches" de tamaños grandes para poder aprender/ajustarse bien:

    * **steps_por_epoch = tamaño_dataset / (batch_size × grad_accumulation)**
  * El **batch_size** es la cantidad de datos que se usan en propagación hacia adelante (feedforward) y que calcula los errores durante el entrenamiento.
  * El **grad_accumulation** es la cantidad de veces que se calculan los gradientes con propagación hacia atrás (backpropagation), antes de realizar un ajuste (actualización) de los pesos durante el entrenamiento.


In [46]:
# Conjuntamos todo para pasar al entrenamiento con los datos de noticias CNN
# y con con ajuste fino del modelo LLM:

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,   # RAM ... baja el batch si tienes poca RAM
    gradient_accumulation_steps=8,   # RAM ... sube el grad_accumulation si tienes poca RAM
    num_train_epochs=1,   # Usualmente 1, 2 o máximo 3 épocas, para no perder capacidad del LLM.
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    #evaluation_strategy="steps",  # Dependiendo la versión de "transformers" que tengas
    eval_strategy="steps",   # instalada, una de estas opciones podrá ser la correcta.
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,   # datos de entrenamiento
    eval_dataset=val_data       # datos de validación
)

trainer.train()

Step,Training Loss,Validation Loss
7,No log,nan


TrainOutput(global_step=7, training_loss=3.546940394810268, metrics={'train_runtime': 126.9859, 'train_samples_per_second': 0.394, 'train_steps_per_second': 0.055, 'total_flos': 1092720839884800.0, 'train_loss': 3.546940394810268, 'epoch': 1.0})

In [52]:
def summarize_article(article_text, model, tokenizer, max_new_tokens=150):

    # Si deseas controlar mejor la salida, le puedes pedir que
    # te proporcione por ejemplo exactamente 3 bullets:
    prompt = f"""Summarize the following news article into bullet points.

Rules:
- Each bullet must be concise
- No repetition
- Max 20 words per bullet


Article:
{article_text}

Highlights:
-"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=768
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        #do_sample=True,
        #temperature=0.7,
        #top_p=0.9,
        #repetition_penalty=1.1,  # >1 reduce repetición de tokens:hola,hola... <1 la favorece.
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extraer solo la parte del resumen
    summary = generated_text.split("Highlights:")[-1].strip()

    return summary


In [ ]:
new_article = """Elon Musk waited too long to sue OpenAI and its leaders, a jury in Oakland, California, decided on Monday. A lawsuit he filed was barred by the statute of limitations, the jury found after about 90 minutes of deliberation.
Their verdict was advisory, but Judge Yvonne Gonzalez Rogers said she agrees with the jury. “The court now confirms the prior indication that it would accept the jury’s findings as its own,” Rogers said. “I think that there’s a substantial amount of evidence to support the jury’s finding, which is why I was prepared to dismiss on the spot,” she added in court on Monday.
Musk helped cofound and fund OpenAI, giving $38 million in its early years. He sued CEO Sam Altman, company president Greg Brockman, and OpenAI in February 2024, alleging that they “stole a charity” and unjustly enriched themselves﻿ when they shifted to a structure that includes a for-profit arm. “I was a fool,” Musk told the court earlier this month. “I gave them free funding to create a startup.”
Musk’s case threatened to derail the ChatGPT maker as it plans what could be a blockbuster IPO. The jury’s decision is a win for OpenAI and its founders, Altman and Brockman.
OpenAI’s attorneys argued the company’s mission hasn’t changed, that it’s still run by a non-profit foundation board, and that Musk waited to file the suit until he founded his own competing artificial intelligence company, xAI. The jury agreed, finding that Musk was aware of the behavior discussed in the lawsuit as early as 2021.
“The finding of the jury confirms that what this lawsuit was was a hypocritical attempt to sabotage a competitor,” William Savitt, OpenAI’s attorney, said on Monday after the verdict. “The fact is that OpenAI is a not-for-profit, mission-driven organization that has been and will continue to be faithful to that mission.”
"""

summary = summarize_article(new_article, model, tokenizer)

print(summary)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


- Elon Musk filed a lawsuit against OpenAI's leaders in February 2024
- Jury found the lawsuit barred by the statute of limitations
- Judge Yvonne Gonzalez Rogers concurred with the jury's decision
- OpenAI and its founders were cleared of wrongdoing


In [53]:
# Incluyendo en el prompt que el resumen lo haga con exactamente 3 frases y Mistral-7B-Instruct:

new_article = """Elon Musk waited too long to sue OpenAI and its leaders, a jury in Oakland, California, decided on Monday. A lawsuit he filed was barred by the statute of limitations, the jury found after about 90 minutes of deliberation.
Their verdict was advisory, but Judge Yvonne Gonzalez Rogers said she agrees with the jury. “The court now confirms the prior indication that it would accept the jury’s findings as its own,” Rogers said. “I think that there’s a substantial amount of evidence to support the jury’s finding, which is why I was prepared to dismiss on the spot,” she added in court on Monday.
Musk helped cofound and fund OpenAI, giving $38 million in its early years. He sued CEO Sam Altman, company president Greg Brockman, and OpenAI in February 2024, alleging that they “stole a charity” and unjustly enriched themselves﻿ when they shifted to a structure that includes a for-profit arm. “I was a fool,” Musk told the court earlier this month. “I gave them free funding to create a startup.”
Musk’s case threatened to derail the ChatGPT maker as it plans what could be a blockbuster IPO. The jury’s decision is a win for OpenAI and its founders, Altman and Brockman.
OpenAI’s attorneys argued the company’s mission hasn’t changed, that it’s still run by a non-profit foundation board, and that Musk waited to file the suit until he founded his own competing artificial intelligence company, xAI. The jury agreed, finding that Musk was aware of the behavior discussed in the lawsuit as early as 2021.
“The finding of the jury confirms that what this lawsuit was was a hypocritical attempt to sabotage a competitor,” William Savitt, OpenAI’s attorney, said on Monday after the verdict. “The fact is that OpenAI is a not-for-profit, mission-driven organization that has been and will continue to be faithful to that mission.”
"""

summary = summarize_article(new_article, model, tokenizer)

print(summary)

- Elon Musk failed to sue OpenAI and its leaders within the legal timeframe, resulting in the lawsuit being dismissed by a jury in Oakland, California.
- The judge agreed with the jury's verdict, stating that there is substantial evidence supporting the jury's finding that OpenAI's leaders "unjustly enriched" themselves.
- OpenAI's attorneys argued that the company's mission hasn't changed and that Musk waited to file the lawsuit until he founded his own competing AI company.


In [35]:
# De manera sencilla seleccionemos algunos artículos y sus highlights
# del conjunto de Validation para evaluarlos con la métrica Rouge:

preds = []
refs = []

for example in val_raw.select(range(10)):     # puede tardar unos pocos mins con 10 predicciones
    pred = generate_summary(example)   # predcción de los highlights con el modelo ajustado.
    preds.append(pred)
    refs.append(example["highlights"])  # los highlights reales del artículo/example.


## **Evaluciones con las métricas Rouge y Bert_score:**

In [49]:
# Métrica Rouge:

# Cargamos la métricas Rouge:
rouge = evaluate.load("rouge")


# Vamos a estar comparando las predicciones del modelo
# del caso "example" dado, con los Highlights reales del mismo:
def generate_summary(example):
    prompt = f"""Summarize the following news article into bullet-point highlights:

Article:
{example['article']}

Highlights:
"""

    # Tokenizamos la noticia con todo y prompt:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generamos los highlights con nuetro modelo ya ajustado:
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        #temperature=0.7,
        #top_p=0.9,
        pad_token_id=tokenizer.eos_token_id  # Para modelos como Mistral/Llama
    )

    summary = tokenizer.decode(outputs[0], skip_special_tokens=True) # regresamos al texto de letras.
    summary = summary.split("Highlights:")[-1].strip()  # separamos por cada bullet del resumen.

    return summary


In [ ]:
# Evaluación con ROUGE-1:
# < 0.2 malo
# 0.2 - 0.3 aceptable
# 0.3 - 0.4 buena
# > 0.4 muy buena

# ++++++++++++++++++++++++++++++++++++++

# Se esperaría que al menos rouge-1 tuviera un buen desempeño.
# rouge-1 mide intersecciones de 1-gramas (de tokens) entre preds y refs.
# rouge-2 mide intersecciones de bigramas (de tokens) entre preds y refs.
# rouge-L mide el tamaño de la secuencia común más larga de tokens (Longest Common Subsequence, LCS).
# rouge-Lsum calcula el promedio ponderado de todas las LCS encontradas en varios enunciados/párrafos.

results = rouge.compute(predictions=preds, references=refs)
print("Resultados encontrados con las variantes de la métrica ROUGE:")
results

Resultados encontrados con las variantes de la métrica ROUGE:


{'rouge1': np.float64(0.3877281954222047),
 'rouge2': np.float64(0.16160903729501908),
 'rougeL': np.float64(0.25413299649584914),
 'rougeLsum': np.float64(0.358418501990604)}

In [ ]:
# BERT-score
# Compara las predicciones con los highlights reales
# de los vectores embebidos y con estos resultados
# calcula la Precision(P), el Recall(R) y F1-score.
# < 0.75  ... bajo
#  0.75 - 0.80 .... acepable
# 0.80 - 0.# > 0.85  .... muy bueno

# ++++++++++++++++++++++++++++++++++++++++++++++++++++


# Quitamos predicciones vacías para que no truene BERT-score:
non_empty_preds = [p for p in preds if p.strip()]   # Si "p" es una cadena vacía o solo espacios en blanco, p.strip()="" es cadena vacía.
corresponding_refs = [r for p, r in zip(preds, refs) if p.strip()]  # seleccionamos los reales de las predicciones no vacías.

if non_empty_preds:
    # Ahora sí, obtenemos los BERT_score de los vectores embebidos y los reales
    # con las métricas de Precision (P), Recall (R) y F1:
    P, R, F1 = bertscore(
        non_empty_preds,
        corresponding_refs,
        lang="en",
        model_type="roberta-large",
        device="cuda",
        batch_size=16,
        use_fast_tokenizer=False # evita errores con pocos recursos.
    )

    print("BERT_score Precision:", P.mean().item())
    print("BERT_score Recall:", R.mean().item())
    print("BERT_score F1:", F1.mean().item())
else:
    print("Tu modelo quedó subentrenado y todas las predicciones fueron vacías, \npor lo que BERT_score es 0.0 en todas sus opciones: P, R, F1.")85 .... bueno



config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT_score Precision: 0.8706067800521851
BERT_score Recall: 0.8861669301986694
BERT_score F1: 0.8782642483711243


* ### **NOTA: Un último comentario: también se puede usar un modelo más grande y de mayor capacidad que el usado como LLM (por ejemplo, GPT-5, Claude, etc), para que funcione como juez en esta etapa final y evalúe la salida/predicciones/desempeño de tu modelo ajustado.**

# **[Complementario] Incluyo estas salidas de los resultados obtenidos con el Mistral base solamente, sin el "Instruct", los cuales fueron diseñados para generar texto, pero no necesariamente para seguir instrucciones... veamos cómo se comporta...**

In [33]:
# Salida con Mistral-7B base, sin indicarle cuántas frases/bullets:

new_article = """Elon Musk waited too long to sue OpenAI and its leaders, a jury in Oakland, California, decided on Monday. A lawsuit he filed was barred by the statute of limitations, the jury found after about 90 minutes of deliberation.
Their verdict was advisory, but Judge Yvonne Gonzalez Rogers said she agrees with the jury. “The court now confirms the prior indication that it would accept the jury’s findings as its own,” Rogers said. “I think that there’s a substantial amount of evidence to support the jury’s finding, which is why I was prepared to dismiss on the spot,” she added in court on Monday.
Musk helped cofound and fund OpenAI, giving $38 million in its early years. He sued CEO Sam Altman, company president Greg Brockman, and OpenAI in February 2024, alleging that they “stole a charity” and unjustly enriched themselves﻿ when they shifted to a structure that includes a for-profit arm. “I was a fool,” Musk told the court earlier this month. “I gave them free funding to create a startup.”
Musk’s case threatened to derail the ChatGPT maker as it plans what could be a blockbuster IPO. The jury’s decision is a win for OpenAI and its founders, Altman and Brockman.
OpenAI’s attorneys argued the company’s mission hasn’t changed, that it’s still run by a non-profit foundation board, and that Musk waited to file the suit until he founded his own competing artificial intelligence company, xAI. The jury agreed, finding that Musk was aware of the behavior discussed in the lawsuit as early as 2021.
“The finding of the jury confirms that what this lawsuit was was a hypocritical attempt to sabotage a competitor,” William Savitt, OpenAI’s attorney, said on Monday after the verdict. “The fact is that OpenAI is a not-for-profit, mission-driven organization that has been and will continue to be faithful to that mission.”
"""

summary = summarize_article(new_article, model, tokenizer)

print(summary)


- Elon Musk sued OpenAI and its leaders over claims that they "stole a charity"
- Judge Yvonne Gonzalez Rogers said she agrees with the jury's decision
- Musk's case threatened to derail the ChatGPT maker as it plans what could be a blockbuster IPO
- The jury's decision is a win for OpenAI and its founders, Sam Altman and Greg Brockman


In [31]:

# Esta salida es con indicando en el prompt que debe generar el resumen con exactamente 3 bullets
# con el modelo Mistral-7B base solamente.

new_article = """Elon Musk waited too long to sue OpenAI and its leaders, a jury in Oakland, California, decided on Monday. A lawsuit he filed was barred by the statute of limitations, the jury found after about 90 minutes of deliberation.
Their verdict was advisory, but Judge Yvonne Gonzalez Rogers said she agrees with the jury. “The court now confirms the prior indication that it would accept the jury’s findings as its own,” Rogers said. “I think that there’s a substantial amount of evidence to support the jury’s finding, which is why I was prepared to dismiss on the spot,” she added in court on Monday.
Musk helped cofound and fund OpenAI, giving $38 million in its early years. He sued CEO Sam Altman, company president Greg Brockman, and OpenAI in February 2024, alleging that they “stole a charity” and unjustly enriched themselves﻿ when they shifted to a structure that includes a for-profit arm. “I was a fool,” Musk told the court earlier this month. “I gave them free funding to create a startup.”
Musk’s case threatened to derail the ChatGPT maker as it plans what could be a blockbuster IPO. The jury’s decision is a win for OpenAI and its founders, Altman and Brockman.
OpenAI’s attorneys argued the company’s mission hasn’t changed, that it’s still run by a non-profit foundation board, and that Musk waited to file the suit until he founded his own competing artificial intelligence company, xAI. The jury agreed, finding that Musk was aware of the behavior discussed in the lawsuit as early as 2021.
“The finding of the jury confirms that what this lawsuit was was a hypocritical attempt to sabotage a competitor,” William Savitt, OpenAI’s attorney, said on Monday after the verdict. “The fact is that OpenAI is a not-for-profit, mission-driven organization that has been and will continue to be faithful to that mission.”
"""

summary = summarize_article(new_article, model, tokenizer)

print(summary)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


- The jury's decision is a win for OpenAI and its founders, Altman and Brockman.
- "I was a fool," Musk told the court earlier this month. "I gave them free funding to create a startup."
- The jury's decision means Musk's lawsuit against OpenAI and its founders is over.
- The jury found that Musk knew of the behavior described in the lawsuit as early as 2021.
- "The finding of the jury confirms that what this lawsuit was was a hypocritical attempt to sabotage a competitor," said William Savitt, OpenAI's attorney.


* ### **Observamos en la salida anterior que aunque las frases son acordes a la nota, no pudo seguir la instrucción de solamente 3 frases:**

In [36]:


# Evaluación con ROUGE del modelo usando solo el Mistral-7B base:



# Evaluación con ROUGE-1:
# < 0.2 malo
# 0.2 - 0.3 aceptable
# 0.3 - 0.4 buena
# > 0.4 muy buena

# ++++++++++++++++++++++++++++++++++++++

# Se esperaría que al menos rouge-1 tuviera un buen desempeño.
# rouge-1 mide intersecciones de 1-gramas (de tokens) entre preds y refs.
# rouge-2 mide intersecciones de bigramas (de tokens) entre preds y refs.
# rouge-L mide el tamaño de la secuencia común más larga de tokens (Longest Common Subsequence, LCS).
# rouge-Lsum calcula el promedio ponderado de todas las LCS encontradas en varios enunciados/párrafos.

results = rouge.compute(predictions=preds, references=refs)
print("Resultados encontrados con las variantes de la métrica ROUGE:")
results

Resultados encontrados con las variantes de la métrica ROUGE:


{'rouge1': np.float64(0.10298608081849547),
 'rouge2': np.float64(0.038910745969297236),
 'rougeL': np.float64(0.059435779254254896),
 'rougeLsum': np.float64(0.08649648325263981)}

In [ ]:

# Resultados solo con el modelo base de Mistral-7B-v01:


# Quitamos predicciones vacías para que no truene BERT-score:
non_empty_preds = [p for p in preds if p.strip()]   # Si "p" es una cadena vacía o solo espacios en blanco, p.strip()="" es cadena vacía.
corresponding_refs = [r for p, r in zip(preds, refs) if p.strip()]  # seleccionamos los reales de las predicciones no vacías.

if non_empty_preds:
    # Ahora sí, obtenemos los BERT_score de los vectores embebidos y los reales
    # con las métricas de Precision (P), Recall (R) y F1:
    P, R, F1 = bertscore(
        non_empty_preds,
        corresponding_refs,
        lang="en",
        model_type="roberta-large",
        device="cuda",
        batch_size=16,
        use_fast_tokenizer=False # evita errores con pocos recursos.
    )

    print("BERT_score Precision:", P.mean().item())
    print("BERT_score Recall:", R.mean().item())
    print("BERT_score F1:", F1.mean().item())
else:
    print("Tu modelo quedó subentrenado y todas las predicciones fueron vacías, \npor lo que BERT_score es 0.0 en todas sus opciones: P, R, F1.")


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT_score Precision: 0.7796982526779175
BERT_score Recall: 0.8119027018547058
BERT_score F1: 0.7953362464904785


* ### **Observamos que tanto con la métrica ROUGE como con las de BERT-score, los desempeños del modelo base de Mistral son menores que los de Mistral-Instruct, lo cual era de esperarse.**